# Discrete Latent Geometry Demo

Inspection notebook for CLI-generated discrete latent geometry diagnostics. The default preset is the promoted standard VQ tokenizer.

RVQ q2 was evaluated on research branches and is not part of the public baseline.


## Parameters


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

RUN_ANALYSIS = False


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs" / "experiments"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: str | Path) -> Path:
    candidate = Path(path).expanduser()
    if candidate.is_absolute():
        return candidate
    return (REPO_ROOT / candidate).resolve()


def display_path(path: str | Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_json(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())


def print_command(command: list[str]) -> None:
    print(" ".join(shlex.quote(part) for part in command))


def maybe_run(command: list[str], *, enabled: bool, label: str) -> None:
    print(f"{label} command:")
    print_command(command)
    if enabled:
        subprocess.run(command, cwd=REPO_ROOT, check=True)
    else:
        print(f"{label} skipped; set RUN_ANALYSIS=True to execute it.")


print(f"Repository root: {REPO_ROOT}")
PRESET = "standard_vq"
BASE_DATA_DIR = Path("data/processed")

PRESETS = {
    "standard_vq": {
        "label": "Standard VQ promoted baseline",
        "config": Path("configs/experiments/sp500_vix_causal_vq_tokenizer.yaml"),
        "tokenizer_dir": Path(
            "outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0"
        ),
        "token_data_dir": Path(
            "outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16"
        ),
        "output_dir": Path("outputs/latent_geometry/sp500_vix_standard_vq"),
    },
}

if PRESET not in PRESETS:
    raise ValueError(f"Unknown PRESET: {PRESET}")

SELECTED = PRESETS[PRESET]
CONFIG_PATH = SELECTED["config"]
TOKENIZER_DIR = SELECTED["tokenizer_dir"]
TOKEN_DATA_DIR = SELECTED["token_data_dir"]
OUTPUT_DIR = SELECTED["output_dir"]
print(SELECTED["label"])
print("output_dir:", display_path(OUTPUT_DIR))

## Artefact Check


In [ ]:
required_inputs = {
    "CONFIG_PATH": CONFIG_PATH,
    "TOKENIZER_DIR": TOKENIZER_DIR,
    "TOKEN_DATA_DIR": TOKEN_DATA_DIR,
    "BASE_DATA_DIR": BASE_DATA_DIR,
}
missing = {name: path for name, path in required_inputs.items() if not repo_path(path).exists()}
if missing:
    print("Missing required artefacts for a real latent-geometry run:")
    for name, path in missing.items():
        print(f"  - {name}: {display_path(path)}")
else:
    print("All configured inputs are present.")
    for name, path in required_inputs.items():
        print(f"  - {name}: {display_path(path)}")

## Optional Analysis Run


In [ ]:
analysis_command = [
    sys.executable,
    "scripts/analyze_discrete_latent_geometry.py",
    "--config",
    display_path(CONFIG_PATH),
    "--tokenizer-dir",
    display_path(TOKENIZER_DIR),
    "--token-data-dir",
    display_path(TOKEN_DATA_DIR),
    "--output-dir",
    display_path(OUTPUT_DIR),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--plot-voronoi",
]
if missing and RUN_ANALYSIS:
    print("RUN_ANALYSIS=True, but required inputs are missing. Resolve paths first.")
else:
    maybe_run(analysis_command, enabled=RUN_ANALYSIS, label="Latent geometry")

## Numeric Summary


In [ ]:
summary = load_json(OUTPUT_DIR / "codebook_geometry_summary.json")
if summary is None:
    print(
        f"No geometry summary found at {display_path(OUTPUT_DIR / 'codebook_geometry_summary.json')}"
    )
else:
    usage = summary.get("usage", {})
    geometry = summary.get("geometry", {})
    metadata = summary.get("metadata", {})
    display(
        pd.DataFrame([
            {
                "quantizer_type": metadata.get("quantizer_type"),
                "embedding_shape": geometry.get("embedding_shape"),
                "active_codes": usage.get("active_code_count"),
                "perplexity": usage.get("codebook_perplexity"),
                "entropy": usage.get("entropy") or usage.get("index_entropy"),
            }
        ])
    )
    if summary.get("condition_buckets"):
        display(Markdown("### VIX-bucket usage"))
        display(pd.DataFrame(summary["condition_buckets"]))

## Generated Plots


In [ ]:
plot_names = [
    "codebook_projection.png",
    "codebook_usage_projection.png",
    "vix_bucket_code_usage.png",
    "token_trajectory_examples.png",
    "codebook_voronoi.png",
    "codebook_nearest_region.png",
]
rows = []
for plot_name in plot_names:
    plot_path = OUTPUT_DIR / plot_name
    rows.append({
        "plot": plot_name,
        "path": display_path(plot_path),
        "exists": repo_path(plot_path).exists(),
    })
display(pd.DataFrame(rows))
for plot_name in plot_names:
    plot_path = repo_path(OUTPUT_DIR / plot_name)
    if plot_path.exists():
        display(Markdown(f"### `{plot_name}`"))
        display(Image(filename=str(plot_path)))

## Interpretation

RVQ q2 was evaluated on research branches and is not part of the public baseline.
